## 7 - Baseline

Entraînement d'un modèle trivial servant de point de comparaison pour les modèles plus complexes.

- La baseline fixe un seuil minimal : tout modèle plus complexe doit la battre pour justifier son coût.
- Sans point de comparaison, un score MAE ou RMSE n'est pas interprétable en soi.
- `DummyRegressor` : stratégie à choisir entre moyenne et médiane selon la distribution de la cible.

La distribution de `SalePrice` est asymétrique à droite. Prédire la moyenne donnerait une MAE artificiellement élevée. La médiane est la constante mathématique qui minimise la MAE, c'est donc notre stratégie de baseline.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge 
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import sys
sys.path.append("..")
from src.preprocessing import regrouper_categories_rares, traitement_valeurs_incohérentes, zero_vers_nan, Transformation_binaire, remplacer_au_dessus_seuil, fusionner_categories, extraire_dates, imputation_mediane_a_nan

# 1. Chargement
df = pd.read_csv("../data/raw/bluebook-for-bulldozers/TrainAndValid.csv", low_memory=False)
df["saledate"] = pd.to_datetime(df["saledate"])
df = df.sort_values("saledate")

# 2. Nettoyage global
taux_nan = df.isnull().mean()
df = df.drop(columns=taux_nan[taux_nan > 0.7].index)
df = traitement_valeurs_incohérentes(df, "YearMade")
df = zero_vers_nan(df, "MachineHoursCurrentMeter")
df = Transformation_binaire(df, "MachineHoursCurrentMeter", "hours_reported")
df = remplacer_au_dessus_seuil(df, "MachineHoursCurrentMeter", 40000)

# 3. Split temporel
train = df[df["saledate"] < "2012-01-01"].copy()
test = df[df["saledate"] >= "2012-01-01"].copy()

# 4. Transformations sur train/test séparés (pour éviter la fuite de données)
train, test = regrouper_categories_rares(train, test, "Hydraulics", 500)
train, test = regrouper_categories_rares(train, test, "fiProductClassDesc", 500)

mapping_enclosure = {"NO ROPS": "OROPS", "EROPS AC": "EROPS w AC", "None or Unspecified": np.nan}
train = fusionner_categories(train, "Enclosure", mapping_enclosure)
test = fusionner_categories(test, "Enclosure", mapping_enclosure)

In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

# 1. Séparation X et y (en excluant SalePrice et MachineID)
X_train = train.drop(columns=["SalePrice", "MachineID"])
y_train = train["SalePrice"]

X_test = test.drop(columns=["SalePrice", "MachineID"])
y_test = test["SalePrice"]

# 2. Instanciation et entraînement de la baseline
dummy_median = DummyRegressor(strategy="median")
dummy_median.fit(X_train, y_train)

# 3. Prédictions
y_pred_train_dummy = dummy_median.predict(X_train)
y_pred_test_dummy = dummy_median.predict(X_test)

# 4. Évaluation (MAE)
mae_train_dummy = mean_absolute_error(y_train, y_pred_train_dummy)
mae_test_dummy = mean_absolute_error(y_test, y_pred_test_dummy)

print(f"Baseline MAE Train : {mae_train_dummy:.2f}")
print(f"Baseline MAE Test  : {mae_test_dummy:.2f}")

Baseline MAE Train : 16454.19
Baseline MAE Test  : 19697.33


Saledate est au format date hors les modèles que l'on souhaite utiliser n'accepte que des valeurs numériques. On va extraire l'année, le mois ,le trimestre 

In [ ]:
X_train = extraire_dates(X_train, "saledate" )
X_test = extraire_dates(X_test, "saledate" )


On va encoder les variables non numérique afin de pouvoir tester nos différents modèles. Avant cela nous allons effectuer un dernier tri dans celles-ci. 

On supprime les variables redondantes ou encore celle qui sont utilisée comme des identifiants 


In [4]:
colonnes_a_supprimer = [
    'SalesID', 'ModelID', 'datasource', 'auctioneerID', 
    'fiModelDesc', 'fiBaseModel', 'fiSecondaryDesc', 
    'ProductGroupDesc', 'ProductSize'
]

X_train = X_train.drop(columns=colonnes_a_supprimer)
X_test = X_test.drop(columns=colonnes_a_supprimer)



In [5]:
colonnes_texte = X_train.select_dtypes(include=['object']).columns.tolist()
print(colonnes_texte)
print(f"Nombre de colonnes à encoder : {len(colonnes_texte)}")

['fiProductClassDesc', 'state', 'ProductGroup', 'Enclosure', 'Forks', 'Ride_Control', 'Transmission', 'Hydraulics', 'Coupler']
Nombre de colonnes à encoder : 9


C:\Users\Abram\AppData\Local\Temp\ipykernel_18480\1769016324.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_texte = X_train.select_dtypes(include=['object']).columns.tolist()


In [6]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# Etape 8: Modélisation 

## 1 - Régression linéaire

### 1.1 - Encodage des variables 


## Diagnostic et traitement des valeurs manquantes

### Taux de NaN

| Colonne | NaN |
|---|---|
| `MachineHoursCurrentMeter` | 83 % |
| `Ride_Control` | 63 % |
| `Transmission` | 54 % |
| `Forks` | 52 % |
| `Coupler` | 47 % |
| `YearMade` | 9,5 % |
| `Enclosure` | 0,1 % |

Le traitement dépend de l'origine du manque, pas de son volume.

### Variables catégorielles

`Forks`, `Ride_Control`, `Transmission`, `Coupler` : NaN structurel, l'équipement n'existe pas sur ce type d'engin. L'information est déjà portée par `ProductGroup` et `fiProductClassDesc`. Imputation par la modalité `"Non applicable"`.

`Enclosure` : taux résiduel de 0,1 %, assimilable à une erreur de saisie. Imputation par `"Missing"`, modalité distincte puisque la nature du manque diffère.

L'imputation par le mode est écartée : elle attribuerait par exemple des `Forks` à un engin qui n'en possède structurellement pas.

### Variables numériques

Pour chaque groupe `fiProductClassDesc` :

- au moins 500 observations non-NaN sur la variable (train) : médiane du groupe
- sinon : médiane globale (train)

Le seuil de 500 est repris de l'étape 4 (regroupement des catégories rares), ce qui évite d'introduire un second paramètre arbitraire.

Le lissage bayésien vers la médiane globale a été écarté : la pondération linéaire par l'effectif n'est fondée que pour une moyenne, une moyenne pondérée de deux médianes n'étant pas la médiane de la population combinée. La moyenne n'est pas pour autant retenue comme substitut, `MachineHoursCurrentMeter` et `YearMade` restant asymétriques à droite après traitement des valeurs extrêmes (même argument qu'à l'étape 7 pour `SalePrice`).

### Fonction

`imputation_mediane_a_nan(train, test, colonne_groupement, colonne_imputer, seuil)` dans `preprocessing.py` : comptage des non-NaN par catégorie, sélection des catégories au-dessus du seuil, `.map()` vers leur médiane puis `.fillna()` sur la médiane globale pour les autres, application par `.fillna()` sur la colonne cible.

Toutes les médianes sont apprises sur train et réappliquées à test (anti-fuite).

Appliquée à `MachineHoursCurrentMeter` et `YearMade`. Vérification : 0 NaN restants sur `X_train` et `X_test`.

In [7]:
X_train, X_test = imputation_mediane_a_nan(X_train, X_test, "fiProductClassDesc", "MachineHoursCurrentMeter", 500)
X_train, X_test = imputation_mediane_a_nan(X_train, X_test, "fiProductClassDesc", "YearMade", 500)

print(X_train["MachineHoursCurrentMeter"].isna().sum())
print(X_train["YearMade"].isna().sum())
print(X_test["MachineHoursCurrentMeter"].isna().sum())
print(X_test["YearMade"].isna().sum())

0
0
0
0


In [8]:
X_train["Forks"] = X_train["Forks"].fillna("Non applicable")
X_train["Ride_Control"] = X_train["Ride_Control"].fillna("Non applicable")
X_train["Transmission"] = X_train["Transmission"].fillna("Non applicable")
X_train["Coupler"] = X_train["Coupler"].fillna("Non applicable")
X_train["Enclosure"] = X_train["Enclosure"].fillna("Missing")

X_test["Forks"] = X_test["Forks"].fillna("Non applicable")
X_test["Ride_Control"] = X_test["Ride_Control"].fillna("Non applicable")
X_test["Transmission"] = X_test["Transmission"].fillna("Non applicable")
X_test["Coupler"] = X_test["Coupler"].fillna("Non applicable")
X_test["Enclosure"] = X_test["Enclosure"].fillna("Missing")

print(X_train.isna().sum())
print(X_test.isna().sum())

YearMade                    0
MachineHoursCurrentMeter    0
fiProductClassDesc          0
state                       0
ProductGroup                0
Enclosure                   0
Forks                       0
Ride_Control                0
Transmission                0
Hydraulics                  0
Coupler                     0
hours_reported              0
saleyear                    0
salemonth                   0
salequarter                 0
dtype: int64
YearMade                    0
MachineHoursCurrentMeter    0
fiProductClassDesc          0
state                       0
ProductGroup                0
Enclosure                   0
Forks                       0
Ride_Control                0
Transmission                0
Hydraulics                  0
Coupler                     0
hours_reported              0
saleyear                    0
salemonth                   0
salequarter                 0
dtype: int64


In [9]:
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

## Encodage des variables catégorielles

Encodage adapté à la régression linéaire, première étape de la séquence de modélisation. Les arbres utiliseront un Ordinal Encoding, traité séparément.

### One-Hot

`ProductGroup`, `Transmission`, `Hydraulics`, `Forks`, `Ride_Control`, `Coupler` : pas d'ordre naturel exploitable et cardinalité limitée à quelques modalités, le coût en colonnes reste faible.

### Ordinal

`Enclosure` seule : l'ordre de confort est défendable (OROPS < EROPS < EROPS w AC).

`Coupler` avait été envisagée en Ordinal (Manual < Hydraulic), puis écartée. La modalité `"Non applicable"`, issue du traitement des NaN structurels, n'a pas de position cohérente sur cette échelle.

### Frequency

`fiProductClassDesc` (~61 modalités) et `state` (~50 modalités) : réduites chacune à une colonne numérique, sans recours à la cible. Le signal est indirect et modéré, mais le coût en dimensionnalité est nul.

Alternatives écartées pour `fiProductClassDesc` :

- extraction d'une variable numérique de tonnage ou de puissance : l'unité varie selon le type d'engin dans le texte brut (profondeur, puissance, poids), une extraction généralisable est trop complexe
- One-Hot : 61 coefficients supplémentaires, incompatible avec l'objectif d'interprétabilité de la régression linéaire
- Target Encoding : calculable sans fuite en apprenant sur train, mais la feature obtenue serait mécaniquement très corrélée à `SalePrice` et écraserait le poids des autres coefficients

`state` avait d'abord été classée en One-Hot, sur une confusion entre le lien de la variable avec l'engin et son coût en nombre de coefficients. Elle est reclassée par cohérence avec `fiProductClassDesc`, sur le même argument de dimensionnalité.

In [10]:
colonne_a_encoderOHE = ["ProductGroup", "Transmission", "Hydraulics", "Forks", "Ride_Control", "Coupler"]
encodeur_ohe = OneHotEncoder(drop="first", sparse_output=False)
resultat_ohe_train = encodeur_ohe.fit_transform(X_train[colonne_a_encoderOHE])
noms_colonnes_ohe_train= encodeur_ohe.get_feature_names_out()
df_ohe_train = pd.DataFrame(resultat_ohe_train, columns=noms_colonnes_ohe_train, index=X_train.index)
X_train = X_train.drop(columns=colonne_a_encoderOHE)
X_train = pd.concat([X_train, df_ohe_train], axis=1)
X_train.shape

(401125, 36)

In [11]:
resultat_ohe_test = encodeur_ohe.transform(X_test[colonne_a_encoderOHE])
noms_colonnes_ohe_test = encodeur_ohe.get_feature_names_out()
df_ohe_test = pd.DataFrame(resultat_ohe_test, columns=noms_colonnes_ohe_test, index=X_test.index)
X_test = X_test.drop(columns=colonne_a_encoderOHE)
X_test = pd.concat([X_test, df_ohe_test], axis=1)

In [12]:
X_test.shape

(11573, 36)

On impute le mode au données "missing" de enclosure. Au vu de la faible fréquence cela n'impactera pas les coefficients et permettra un encodage ordinal

In [13]:
mode_enclosure = X_train["Enclosure"].mode()[0]
X_train.loc[X_train["Enclosure"]=="Missing","Enclosure"] = mode_enclosure
X_train["Enclosure"].value_counts()

Enclosure
OROPS         174262
EROPS         139026
EROPS w AC     87837
Name: count, dtype: int64

In [14]:
mode_enclosure = X_train["Enclosure"].mode()[0]
X_test.loc[X_test["Enclosure"]=="Missing","Enclosure"] = mode_enclosure
X_test["Enclosure"].value_counts()

Enclosure
EROPS w AC    4782
OROPS         4048
EROPS         2743
Name: count, dtype: int64

In [15]:
encodeur_ordinal = OrdinalEncoder(categories=[["OROPS", "EROPS", "EROPS w AC"]])
resultat_ordinal_train = encodeur_ordinal.fit_transform(X_train[["Enclosure"]])
noms_colonnes_ordinal_train= encodeur_ordinal.get_feature_names_out()
df_ordinal_train = pd.DataFrame(resultat_ordinal_train, columns=noms_colonnes_ordinal_train, index=X_train.index)
X_train = X_train.drop(columns="Enclosure")
X_train = pd.concat([X_train, df_ordinal_train], axis=1)
X_train.shape

(401125, 36)

In [16]:
resultat_ordinal_test = encodeur_ordinal.transform(X_test[["Enclosure"]])
noms_colonnes_ordinal_test= encodeur_ordinal.get_feature_names_out()
df_ordinal_test= pd.DataFrame(resultat_ordinal_test, columns=noms_colonnes_ordinal_train, index=X_test.index)
X_test = X_test.drop(columns="Enclosure")
X_test = pd.concat([X_test, df_ordinal_test], axis=1)
X_test.shape

(11573, 36)

On applique le frequency encodage a state et fiProductClassDesc

In [17]:
Freq = X_train["fiProductClassDesc"].value_counts()
X_train["fiProductClassDesc_freq"] = X_train["fiProductClassDesc"].map(Freq)
X_train[["fiProductClassDesc", "fiProductClassDesc_freq"]].head(10)


,fiProductClassDesc,fiProductClassDesc_freq
205615,"Track Type Tractor, Dozer - 105.0 to 130.0 Hor...",4877
274835,Wheel Loader - 120.0 to 135.0 Horsepower,10551
141296,"Track Type Tractor, Dozer - 190.0 to 260.0 Hor...",6557
212552,Other,2344
62755,"Track Type Tractor, Dozer - 20.0 to 75.0 Horse...",17788
54653,"Track Type Tractor, Dozer - 130.0 to 160.0 Hor...",11140
81383,Wheel Loader - 60.0 to 80.0 Horsepower,4669
204924,Wheel Loader - 90.0 to 100.0 Horsepower,3688
135376,"Track Type Tractor, Dozer - 130.0 to 160.0 Hor...",11140
113390,Motorgrader - 45.0 to 130.0 Horsepower,7096


In [18]:
Freq = X_train["fiProductClassDesc"].value_counts()
X_test["fiProductClassDesc_freq"] = X_test["fiProductClassDesc"].map(Freq)
X_test[["fiProductClassDesc", "fiProductClassDesc_freq"]].head(100)

,fiProductClassDesc,fiProductClassDesc_freq
405675,"Track Type Tractor, Dozer - 190.0 to 260.0 Hor...",6557
401133,Motorgrader - 130.0 to 145.0 Horsepower,5487
406076,Backhoe Loader - 15.0 to 16.0 Ft Standard Digg...,10566
409018,"Hydraulic Excavator, Track - 12.0 to 14.0 Metr...",11354
409026,"Hydraulic Excavator, Track - 14.0 to 16.0 Metr...",4554
...,...,...
401143,"Hydraulic Excavator, Track - 28.0 to 33.0 Metr...",6134
401140,Wheel Loader - 135.0 to 150.0 Horsepower,4913
401190,"Hydraulic Excavator, Track - 50.0 to 66.0 Metr...",1702
401165,Wheel Loader - 275.0 to 350.0 Horsepower,5308


In [19]:
Freq = X_train["state"].value_counts()
X_train["state_freq"] = X_train["state"].map(Freq)
X_train[["state", "state_freq"]].head(10)

,state,state_freq
205615,Texas,51682
274835,Florida,63944
141296,Florida,63944
212552,Florida,63944
62755,Florida,63944
54653,Florida,63944
81383,Florida,63944
204924,Florida,63944
135376,Florida,63944
113390,Florida,63944


In [20]:
Freq = X_train["state"].value_counts()
X_test["state_freq"] = X_test["state"].map(Freq)
X_test[["state", "state_freq"]].head(10)

,state,state_freq
405675,Michigan,1763
401133,Florida,63944
406076,Iowa,1215
409018,Mississippi,12961
409026,Mississippi,12961
401129,Florida,63944
411231,Mississippi,12961
401138,Utah,2895
401137,West Virginia,746
401136,West Virginia,746


In [21]:
Colonne_a_drop=["state","fiProductClassDesc"]
X_train = X_train.drop(columns=Colonne_a_drop)
X_test= X_test.drop(columns=Colonne_a_drop)

In [22]:
X_train.shape
X_test.shape
X_train.columns.tolist()

['YearMade',
 'MachineHoursCurrentMeter',
 'hours_reported',
 'saleyear',
 'salemonth',
 'salequarter',
 'ProductGroup_MG',
 'ProductGroup_SSL',
 'ProductGroup_TEX',
 'ProductGroup_TTT',
 'ProductGroup_WL',
 'Transmission_Autoshift',
 'Transmission_Direct Drive',
 'Transmission_Hydrostatic',
 'Transmission_Non applicable',
 'Transmission_None or Unspecified',
 'Transmission_Powershift',
 'Transmission_Powershuttle',
 'Transmission_Standard',
 'Hydraulics_3 Valve',
 'Hydraulics_4 Valve',
 'Hydraulics_Auxiliary',
 'Hydraulics_Base + 1 Function',
 'Hydraulics_Other',
 'Hydraulics_Standard',
 'Forks_None or Unspecified',
 'Forks_Yes',
 'Ride_Control_Non applicable',
 'Ride_Control_None or Unspecified',
 'Ride_Control_Yes',
 'Coupler_Manual',
 'Coupler_Non applicable',
 'Coupler_None or Unspecified',
 'Enclosure',
 'fiProductClassDesc_freq',
 'state_freq']

In [23]:
X_train["Enclosure"].dtype

dtype('float64')

In [24]:
X_train.columns.tolist()

['YearMade',
 'MachineHoursCurrentMeter',
 'hours_reported',
 'saleyear',
 'salemonth',
 'salequarter',
 'ProductGroup_MG',
 'ProductGroup_SSL',
 'ProductGroup_TEX',
 'ProductGroup_TTT',
 'ProductGroup_WL',
 'Transmission_Autoshift',
 'Transmission_Direct Drive',
 'Transmission_Hydrostatic',
 'Transmission_Non applicable',
 'Transmission_None or Unspecified',
 'Transmission_Powershift',
 'Transmission_Powershuttle',
 'Transmission_Standard',
 'Hydraulics_3 Valve',
 'Hydraulics_4 Valve',
 'Hydraulics_Auxiliary',
 'Hydraulics_Base + 1 Function',
 'Hydraulics_Other',
 'Hydraulics_Standard',
 'Forks_None or Unspecified',
 'Forks_Yes',
 'Ride_Control_Non applicable',
 'Ride_Control_None or Unspecified',
 'Ride_Control_Yes',
 'Coupler_Manual',
 'Coupler_Non applicable',
 'Coupler_None or Unspecified',
 'Enclosure',
 'fiProductClassDesc_freq',
 'state_freq']

On va standardiser les colonnes

In [25]:
Colonne_a_standardiser = ["YearMade", "MachineHoursCurrentMeter", "saleyear", "salemonth", "salequarter", "fiProductClassDesc_freq", "state_freq"]
standardisation = StandardScaler()
Resultat_standardisation =standardisation.fit_transform(X_train[Colonne_a_standardiser])
Resultat_standardisation.shape
X_train[Colonne_a_standardiser] = Resultat_standardisation
Resultat_standardisation =standardisation.transform(X_test[Colonne_a_standardiser])
X_test[Colonne_a_standardiser] = Resultat_standardisation

In [26]:
X_train[Colonne_a_standardiser].describe()

,YearMade,MachineHoursCurrentMeter,saleyear,salemonth,salequarter,fiProductClassDesc_freq,state_freq
count,4.011250e+05,4.011250e+05,4.011250e+05,4.011250e+05,4.011250e+05,4.011250e+05,4.011250e+05
mean,1.030288e-14,-5.214928e-17,-1.726821e-14,1.201701e-16,1.587152e-16,1.983940e-17,6.121871e-17
std,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00
min,-7.920263e+00,-1.413666e+00,-2.623435e+00,-1.578894e+00,-1.217516e+00,-8.532605e-01,-1.112318e+00
25%,-6.236146e-01,-5.207369e-01,-7.117827e-01,-9.948794e-01,-1.217516e+00,-5.417481e-01,-7.028246e-01
50%,2.348146e-01,-3.529704e-01,3.309369e-01,-1.188573e-01,-3.385489e-01,-3.740377e-01,-5.336871e-01
75%,7.713329e-01,3.898244e-01,8.522968e-01,7.571648e-01,5.404182e-01,-2.064179e-02,1.195239e+00
max,1.951673e+00,1.220110e+01,1.199870e+00,1.633187e+00,1.419385e+00,2.413361e+00,1.742747e+00


## Standardisation

Objectif : ramener les variables numériques à large échelle sur une base comparable avant l'entraînement de la régression linéaire.

Sans standardisation, l'amplitude d'une variable détermine mécaniquement l'ordre de grandeur de son coefficient : `fiProductClassDesc_freq` monte jusqu'à ~8750 et reçoit donc un coefficient très petit, quand une variable binaire en reçoit un bien plus grand. L'écart ne reflète pas l'importance réelle des variables, seulement leur unité, ce qui rend les coefficients ininterprétables entre eux.

**Colonnes standardisées** : `YearMade`, `MachineHoursCurrentMeter`, `saleyear`, `salemonth`, `salequarter`, `fiProductClassDesc_freq`, `state_freq`.

**Colonnes exclues** : les binaires issues du One-Hot et l'ordinal `Enclosure` (0/1/2), déjà sur une échelle homogène.

Méthode : `StandardScaler` (centrer-réduire). `fit_transform` sur train, `transform` seul sur test, pour éviter toute fuite.

Vérification : moyennes proches de 0 et écarts-types proches de 1 sur les 7 colonnes concernées.

In [27]:
RegLinear = LinearRegression()
RegLinear.fit(X_train, y_train) 
y_pred_train_reglinear = RegLinear.predict(X_train)
y_pred_test_reglinear = RegLinear.predict(X_test)
mae_train_reglinear = mean_absolute_error(y_train, y_pred_train_reglinear)
mae_test_reglinear = mean_absolute_error(y_test, y_pred_test_reglinear)
print(f"Régression linéaire MAE Train : {mae_train_reglinear:.2f}")
print(f"Régression linéaire MAE Test  : {mae_test_reglinear:.2f}")

Régression linéaire MAE Train : 11619.26
Régression linéaire MAE Test  : 14354.30


## Régression linéaire

Entraînement de `LinearRegression` sur X_train encodé et standardisé.

| Modèle | MAE train | MAE test |
|---|---|---|
| Baseline (médiane) | 16 454,19 | 19 697,33 |
| Régression linéaire | 11 619,26 | 14 354,30 |

Nette amélioration sur la baseline. L'écart relatif train→test est un peu plus élevé (~23,5 % contre ~19,7 %), cohérent avec la capacité d'ajustement du modèle : 15 features contre aucune.

Examen rapide des coefficients : quelques valeurs extrêmes sur les modalités `"Non applicable"` (ex: `Transmission_Non applicable` à -38 874), signe d'une colinéarité structurelle entre variables issues du même sous-ensemble de NaN. Sans impact démontré sur la MAE  non creusé davantage, l'objectif de ce modèle étant comparatif, pas inférentiel.

## Ridge

Régularisation L2 : la pénalité $\alpha \sum_j \beta_j^2$ contraint la taille des coefficients, utile en cas de colinéarité. MAE quasi identique à la régression linéaire (11 619,24 / 14 354,60), confirmant que la colinéarité affecte la stabilité des coefficients, pas la performance prédictive.

Standardisation réutilisée telle quelle (indispensable, la pénalité portant sur la valeur des coefficients).

In [28]:
RegRidge = Ridge(alpha=1.0)
RegRidge.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [29]:
y_pred_train_ridge =RegRidge.predict(X_train)
y_pred_test_ridge = RegRidge.predict(X_test)
mae_train_ridge= mean_absolute_error(y_train, y_pred_train_ridge) 
mae_test_ridge = mean_absolute_error(y_test, y_pred_test_ridge) 
print(f"Régression ridge Train : {mae_train_ridge:.2f}")
print(f"Régression ridge Test  : {mae_test_ridge:.2f}")

Régression ridge Train : 11619.24
Régression ridge Test  : 14354.60


## Point de sauvegarde intermédiaire

`X_train_clean` et `X_test_clean` sont sauvegardés juste après le traitement des NaN, avant tout encodage. Chaque modèle de la séquence repart d'une copie de ces jeux, sans rejouer le nettoyage et sans altérer le pivot.

L'encodage devient ainsi une décision propre à chaque famille de modèles :

- **linéaire et Ridge** : One-Hot, Frequency, puis standardisation, la géométrie du modèle dépendant de l'échelle des variables
- **arbres** : Ordinal Encoding simple sur toutes les catégorielles, sans standardisation, les découpages ne portant que sur l'ordre des valeurs et non sur leurs écarts

In [30]:
X_train_arbre = X_train_clean.copy()
X_test_arbre = X_test_clean.copy()


Encodage Ordinal

In [31]:
colonnes_a_encoder_arbre = ["fiProductClassDesc", "state", "ProductGroup", "Enclosure", "Forks", "Ride_Control", "Transmission", "Hydraulics", "Coupler"]

In [32]:
encodeur_ordinal = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1)
resultat_ordinal_train = encodeur_ordinal.fit_transform(X_train_arbre[colonnes_a_encoder_arbre])
noms_colonnes_ordinal_train= encodeur_ordinal.get_feature_names_out()
df_ordinal_train = pd.DataFrame(resultat_ordinal_train, columns=noms_colonnes_ordinal_train, index=X_train_arbre.index)
X_train_arbre = X_train_arbre.drop(columns=colonnes_a_encoder_arbre)
X_train_arbre = pd.concat([X_train_arbre, df_ordinal_train], axis=1)
X_train_arbre.shape


(401125, 15)

In [33]:
resultat_ordinal_test = encodeur_ordinal.transform(X_test_arbre[colonnes_a_encoder_arbre])
noms_colonnes_ordinal_test= encodeur_ordinal.get_feature_names_out()
df_ordinal_test = pd.DataFrame(resultat_ordinal_test, columns=noms_colonnes_ordinal_test, index=X_test_arbre.index)
X_test_arbre = X_test_arbre.drop(columns=colonnes_a_encoder_arbre)
X_test_arbre = pd.concat([X_test_arbre, df_ordinal_test], axis=1)
X_test_arbre.shape


(11573, 15)

## Arbre de décision

### Préparation

`X_train_clean` et `X_test_clean` sont copiés en `X_train_arbre` et `X_test_arbre`, avec Ordinal Encoding simple sur toutes les catégorielles. Ni One-Hot, ni Frequency, ni standardisation : les découpages ne portent que sur l'ordre des valeurs, pas sur leurs écarts.

**Bug corrigé en amont.** `df` n'était pas trié chronologiquement dès le chargement, ce qui invalidait `TimeSeriesSplit` : le découpage se faisait par position de ligne et non par date réelle. Correction par `df.sort_values("saledate")` en tête de pipeline, propagée à toute la suite.



In [34]:
decisiontree = DecisionTreeRegressor(criterion="absolute_error") 
decisiontree.fit(X_train_arbre, y_train)
y_pred_train_tree = decisiontree.predict(X_train_arbre)
y_pred_test_tree = decisiontree.predict(X_test_arbre)
mae_train_tree = mean_absolute_error(y_train,y_pred_train_tree)
mae_test_tree = mean_absolute_error(y_test,y_pred_test_tree)
print(f"Arbre de décision MAE Train : {mae_train_tree:.2f}")
print(f"Arbre de décision MAE Test  : {mae_test_tree:.2f}")



Arbre de décision MAE Train : 723.09
Arbre de décision MAE Test  : 11066.59


Surapprentissage manifeste : l'arbre non contraint mémorise le jeu d'entraînement.

Recherche des hyperparamètres de régularisation par `GridSearchCV` :



In [35]:
param_grid = {
    "max_depth": [3, 5, 10, 15],
    "min_samples_leaf": [10, 50, 100],
    "min_samples_split": [20, 100, 200]
}
tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(criterion="absolute_error"),
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error"
)

grid_search.fit(X_train_arbre, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeR...solute_error')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [3, 5, ...], 'min_samples_leaf': [10, 50, ...], 'min_samples_split': [20, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for cal

In [36]:
best_tree = grid_search.best_estimator_

y_pred_train_tree_opt = best_tree.predict(X_train_arbre)
y_pred_test_tree_opt = best_tree.predict(X_test_arbre)

mae_train_tree_opt = mean_absolute_error(y_train, y_pred_train_tree_opt)
mae_test_tree_opt = mean_absolute_error(y_test, y_pred_test_tree_opt)

print(f"Arbre optimisé MAE Train : {mae_train_tree_opt:.2f}")
print(f"Arbre optimisé MAE Test  : {mae_test_tree_opt:.2f}")

Arbre optimisé MAE Train : 6408.84
Arbre optimisé MAE Test  : 9163.04


### Recherche d'hyperparamètres

L'estimateur utilise `criterion="absolute_error"`, cohérent avec la MAE choisie comme métrique d'évaluation depuis la Baseline : ce critère optimise la médiane dans chaque feuille, comme la médiane minimise la MAE globale.

`GridSearchCV` sur `max_depth`, `min_samples_leaf` et `min_samples_split`, scoring `neg_mean_absolute_error`, validation par `TimeSeriesSplit(n_splits=5)`. Les plis respectent l'ordre chronologique : un `KFold` classique entraînerait le modèle sur des ventes postérieures à celles qu'il évalue, reproduisant à l'intérieur de la validation la fuite temporelle écartée au moment du split.

Meilleure combinaison : `max_depth=15`, `min_samples_split=20`, `min_samples_leaf=10`.

### Résultats

| Modèle | MAE train | MAE test |
|---|---|---|
| Baseline (médiane) | 16 454,19 | 19 697,33 |
| Régression linéaire | 11 619,26 | 14 354,30 |
| Arbre sans contrainte | 777,12 | 11 191,30 |
| Arbre optimisé | 6 409,59 | 9 158,87 |

L'écart train/test est nettement resserré par rapport à l'arbre libre, signe d'une meilleure généralisation. `X_test_arbre` n'a jamais été vu par la recherche d'hyperparamètres.

Meilleure performance de la séquence à ce stade, MAE test inférieure d'environ 36 % à celle de la régression linéaire. La relation entre les variables et `SalePrice` comporte donc une composante non linéaire significative, ce qui justifie de poursuivre vers Random Forest et Gradient Boosting.

La grille testée ici a été fixée empiriquement plutôt que déterminée par une méthode rigoureuse, signal à noter : `max_depth=15` correspond à la borne haute testée, suggérant un optimum possiblement situé au-delà. Pour Random Forest, une recherche en deux temps (grossière puis fine, avec vérification systématique des bornes) sera appliquée.

## Random Forest

### Préparation

Copies dédiées `X_train_arbre_aleatoire` et `X_test_arbre_aleatoire`, distinctes de celles de l'arbre simple.

L'encodeur `encodeur_ordinal` déjà ajusté sur l'arbre de décision est réutilisé par `.transform()` seul, sans réentraînement. Les correspondances catégorie→nombre restent ainsi identiques entre les deux modèles, ce qui rend leurs performances comparables et écarte le risque qu'une catégorie absente d'un sous-échantillon reçoive un code différent.

### Échantillonnage pour la recherche grossière

La recherche d'hyperparamètres sur l'intégralité du train est trop coûteuse. Elle est menée sur un échantillon de 50 000 lignes tiré dans `X_train_arbre_aleatoire` (`random_state=42`), avec `y_train_sample` aligné sur les mêmes index.

**Point de vigilance.** `.sample()` casse l'ordre chronologique dont `TimeSeriesSplit` dépend : les plis seraient constitués sur un ordre aléatoire et la validation temporelle perdrait tout sens. Correction par `.sort_index()` après tirage, l'index ayant conservé l'ordre chronologique hérité du tri initial de `df`.

### Critère

`RandomForestRegressor` accepte `criterion="absolute_error"`, retenu pour rester aligné sur la MAE, métrique de référence du projet.

In [ ]:
X_train_arbre_aleatoire= X_train_clean.copy()
X_test_arbre_aleatoire = X_test_clean.copy()

On réutilise l'encodeur de l'arbre de décisio, pas besoin de réapprendre car il connait déjà toutes les catégorie

In [38]:
resultat_ordinal_train = encodeur_ordinal.transform(X_train_arbre_aleatoire[colonnes_a_encoder_arbre])
noms_colonnes_ordinal_train= encodeur_ordinal.get_feature_names_out()
df_ordinal_train = pd.DataFrame(resultat_ordinal_train, columns=noms_colonnes_ordinal_train, index=X_train_arbre_aleatoire.index)
X_train_arbre_aleatoire = X_train_arbre_aleatoire.drop(columns=colonnes_a_encoder_arbre)
X_train_arbre_aleatoire = pd.concat([X_train_arbre_aleatoire, df_ordinal_train], axis=1)
X_train_arbre_aleatoire.shape

(401125, 15)

In [39]:
resultat_ordinal_test = encodeur_ordinal.transform(X_test_arbre_aleatoire[colonnes_a_encoder_arbre])
noms_colonnes_ordinal_test= encodeur_ordinal.get_feature_names_out()
df_ordinal_test = pd.DataFrame(resultat_ordinal_test, columns=noms_colonnes_ordinal_test, index=X_test_arbre_aleatoire.index)
X_test_arbre_aleatoire = X_test_arbre_aleatoire.drop(columns=colonnes_a_encoder_arbre)
X_test_arbre_aleatoire = pd.concat([X_test_arbre_aleatoire, df_ordinal_test], axis=1)
X_test_arbre_aleatoire.shape

(11573, 15)

In [40]:
X_train_sample = X_train_arbre_aleatoire.sample(n=50000, random_state=42).sort_index()
y_train_sample = y_train.loc[X_train_sample.index]


### Grille d'hyperparamètres

`n_estimators` fixé à 100, hors grille. Augmenter le nombre d'arbres réduit mécaniquement la variance de l'agrégat sans provoquer de surapprentissage, contrairement aux autres hyperparamètres : il n'y a donc pas d'optimum à chercher, seulement une valeur suffisante à choisir.

Paramètres testés :

- `max_depth` [10, 20, 30] : plage large, l'arbre seul ayant retenu `max_depth=15`, soit la borne haute de sa grille, ce qui suggère que l'optimum se situe peut-être plus loin
- `max_features` [3, 5, 8] : centré sur la règle empirique $p/3$, soit environ 5 pour 15 variables, avec une valeur de part et d'autre pour vérifier si cette règle se confirme sur ce dataset

`min_samples_leaf` et `min_samples_split` sont volontairement exclus. Chaque arbre est entraîné sur un échantillon bootstrap différent, et la moyenne de leurs prédictions annule une partie du bruit individuel : contraindre chaque arbre pris isolément devient moins nécessaire que sur un arbre unique.

Cette recherche sur 50 000 lignes ne vise qu'une première estimation grossière, à affiner ensuite sur une grille resserrée puis, si nécessaire, sur l'ensemble du train.

In [41]:
param_grid = {
    "max_depth": [10, 20, 30],
    "max_features": [3, 5, 8],
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(criterion="absolute_error", n_estimators=100),
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error"
)

grid_search.fit(X_train_sample, y_train_sample)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...solute_error')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [10, 20, ...], 'max_features': [3, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int

In [42]:
param_grid = {
    "max_depth": [15, 20, 25],
    "max_features": [8, 10, 12],
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search2 = GridSearchCV(
    estimator=RandomForestRegressor(criterion="absolute_error", n_estimators=100),
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error"
)

grid_search2.fit(X_train_sample, y_train_sample)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...solute_error')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [15, 20, ...], 'max_features': [8, 10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: in

### Optimisation des hyperparamètres du Random Forest

* **Paramètres fixés hors grille** : `n_estimators` maintenu à 100 et exclusion de `min_samples_leaf` / `min_samples_split` (le bagging du Random Forest lissant nativement le bruit des arbres individuels).
* **Résultats de la grille fine** (sur échantillon 50k) : `best_score_` obtenu de -9147 (MAE estimée à 9 147 $) pour `best_params_ = {'max_depth': 15, 'max_features': 10}`.
* **Analyse du signal de bordure** : La valeur retenue `max_depth=15` constitue la borne basse de la grille testée (`[15, 20, 25]`).
* **Arrêt assumé de l'optimisation (Arbitrage Coût/Bénéfice)** : Refus conscient d'explorer plus bas (ex: `[12, 15, 17]`). Compte tenu de la robustesse intrinsèque du modèle et du biais lié à l'échantillon de 50 000 lignes, une micro-optimisation supplémentaire capterait davantage de bruit statistique qu'un signal généralisable aux 400 000 lignes.
* **Paramètres finaux retenus pour l'étape 3** : L'entraînement final sur le jeu complet s'effectuera avec `max_depth=15` et `max_features=10`.

### Recherche d'hyperparamètres

Une première grille grossière (`max_depth` [10, 20, 30], `max_features` [3, 5, 8]) retient `max_depth=20` et `max_features=8`, pour un score de validation d'environ -9173.

La grille fine (`max_depth` [15, 20, 25], `max_features` [8, 10, 12]) retient `max_depth=15` et `max_features=10`, pour un score d'environ -9147. Le gain est de 26 sur un score d'environ 9 150, soit 0,3 %, ordre de grandeur du bruit d'échantillonnage : la recherche a atteint un plateau.

`max_depth=15` se situe sur la borne basse de la grille fine, signal habituel qu'un optimum pourrait exister en deçà. La recherche s'arrête néanmoins ici, par arbitrage coût/bénéfice : la Random Forest est peu sensible à un réglage exact de profondeur, et la recherche porte sur un échantillon de 50 000 lignes contre 401 125 au total, ce qui limite de toute façon la précision atteignable sur ce paramètre.

### Modèle final

Entraînement sur l'ensemble complet du train (401 125 lignes), avec `random_state=42` pour la reproductibilité. L'écart avec le run sans graine est négligeable (5 869,10 → 5 869,11 en train, 8 642,02 → 8 642,01 en test), ce qui confirme empiriquement la stabilité de l'agrégat.

| Modèle | MAE train | MAE test |
|---|---|---|
| Arbre optimisé | 6 409,59 | 9 158,87 |
| Random Forest | 5 869,11 | 8 642,01 |

Amélioration de 516,86 $ sur le test, soit environ 5,6 %.

## Gradient Boosting

Recherche d'hyperparamètres en deux temps sur un échantillon de 50 000 lignes, avec `TimeSeriesSplit` (5 folds). Le nombre d'arbres optimal à chaque combinaison est lu via `staged_predict()`, qui donne l'erreur à chaque étape de la séquence sans réentraîner le modèle.

`n_iter_no_change`, l'early stopping natif de `GradientBoostingRegressor`, est écarté : il repose en interne sur un split aléatoire (`shuffle=True`), incompatible avec l'ordre chronologique des données.

Grille grossière `learning_rate` [0.01, 0.1, 0.2] / `max_depth` [3, 4, 5] : meilleure combinaison `lr=0.1`, `depth=5`, `n=432`, MAE ≈ 9233,55.

Grille fine `learning_rate` [0.05, 0.1, 0.15] / `max_depth` [5, 6, 7] : meilleure combinaison `lr=0.1`, `depth=7`, `n=454`, MAE ≈ 9005,04.

`max_depth=7` se situe en bordure haute de la grille. La recherche n'est pas étendue à 8 et au-delà : en Gradient Boosting, contrairement au Random Forest, des arbres individuels trop complexes combinés à une longue séquence corrective amplifient le risque d'apprendre du bruit plutôt que du signal.

In [ ]:
X_train_arbre_gradientboosting = X_train_clean.copy()
X_test_arbre_gradientboosting = X_test_clean.copy()


In [45]:
colonnes_a_encoder_arbre = ["fiProductClassDesc", "state", "ProductGroup", "Enclosure", "Forks", "Ride_Control", "Transmission", "Hydraulics", "Coupler"]
resultat_ordinal_train = encodeur_ordinal.transform(X_train_arbre_gradientboosting[colonnes_a_encoder_arbre])
noms_colonnes_ordinal_train= encodeur_ordinal.get_feature_names_out()
df_ordinal_train = pd.DataFrame(resultat_ordinal_train, columns=noms_colonnes_ordinal_train, index=X_train_arbre_gradientboosting.index)
X_train_arbre_gradientboosting = X_train_arbre_gradientboosting.drop(columns=colonnes_a_encoder_arbre)
X_train_arbre_gradientboosting = pd.concat([X_train_arbre_gradientboosting, df_ordinal_train], axis=1)
X_train_arbre_gradientboosting.shape

(401125, 15)

In [46]:
colonnes_a_encoder_arbre = ["fiProductClassDesc", "state", "ProductGroup", "Enclosure", "Forks", "Ride_Control", "Transmission", "Hydraulics", "Coupler"]
resultat_ordinal_train = encodeur_ordinal.transform(X_test_arbre_gradientboosting[colonnes_a_encoder_arbre])
noms_colonnes_ordinal_train= encodeur_ordinal.get_feature_names_out()
df_ordinal_train = pd.DataFrame(resultat_ordinal_train, columns=noms_colonnes_ordinal_train, index=X_test_arbre_gradientboosting.index)
X_test_arbre_gradientboosting = X_test_arbre_gradientboosting.drop(columns=colonnes_a_encoder_arbre)
X_test_arbre_gradientboosting = pd.concat([X_test_arbre_gradientboosting, df_ordinal_train], axis=1)
X_test_arbre_gradientboosting.shape

(11573, 15)

In [47]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

X_train_sample_gb = X_train_arbre_gradientboosting.sample(n=50000, random_state=42).sort_index()
y_train_sample_gb = y_train.loc[X_train_sample_gb.index]

# 3. Définition de la grille et de la validation
learning_rates = [0.05, 0.1, 0.15]
max_depths = [5, 6, 7]
n_estimators_max = 500
n_splits = 5

tscv = TimeSeriesSplit(n_splits=n_splits)

# Dictionnaires pour stocker les résultats
resultats_grille = {}
meilleur_score_global = float('inf')
meilleurs_params_globaux = {}

print("Début de la recherche sur grille avec TimeSeriesSplit et staged_predict()...")

# 4. Boucle externe sur les combinaisons (learning_rate, max_depth)
for lr in learning_rates:
    for depth in max_depths:
        print(f"\nTest combinaison : learning_rate={lr}, max_depth={depth}")
        
        # Tableau pour cumuler les MAE à chaque étape pour les 5 folds
        mae_cumulee_folds = np.zeros(n_estimators_max)
        
        # 5. Boucle interne sur les folds TimeSeriesSplit
        for fold, (train_index, val_index) in enumerate(tscv.split(X_train_sample_gb)):
            # Extraction des données du fold
            X_fold_train = X_train_sample_gb.iloc[train_index]
            X_fold_val = X_train_sample_gb.iloc[val_index]
            y_fold_train = y_train_sample_gb.iloc[train_index]
            y_fold_val = y_train_sample_gb.iloc[val_index]
            
            # Initialisation et entraînement (UNE seule fois par fold avec 500 arbres)
            modele_gb = GradientBoostingRegressor(
                loss='absolute_error',
                learning_rate=lr,
                max_depth=depth,
                n_estimators=n_estimators_max,
                random_state=42
            )
            modele_gb.fit(X_fold_train, y_fold_train)
            
            # 6. Évaluation itérative avec staged_predict()
            for i, y_pred_stage in enumerate(modele_gb.staged_predict(X_fold_val)):
                mae_cumulee_folds[i] += mean_absolute_error(y_fold_val, y_pred_stage)
            
        # 7. Moyenne des courbes sur les 5 folds
        mae_moyenne_cv = mae_cumulee_folds / n_splits
        
        # 8. Extraction du meilleur n_estimators pour cette combinaison
        # argmin() donne l'index (0 à 499), on ajoute 1 pour avoir le nombre d'arbres (1 à 500)
        best_n = np.argmin(mae_moyenne_cv) + 1
        best_mae = np.min(mae_moyenne_cv)
        
        resultats_grille[(lr, depth)] = (best_n, best_mae)
        print(f"-> Meilleur n_estimators : {best_n} | MAE moyenne = {best_mae:.2f}")
        
        # Mise à jour du meilleur modèle global
        if best_mae < meilleur_score_global:
            meilleur_score_global = best_mae
            meilleurs_params_globaux = {
                'learning_rate': lr, 
                'max_depth': depth, 
                'n_estimators': best_n
            }

# 9. Affichage du grand gagnant
print("\n" + "="*50)
print("RÉSULTAT FINAL SUR L'ÉCHANTILLON (50k lignes) :")
print(f"Meilleurs hyperparamètres : {meilleurs_params_globaux}")
print(f"Meilleure MAE associée : {meilleur_score_global:.2f}")
print("="*50)

Début de la recherche sur grille avec TimeSeriesSplit et staged_predict()...

Test combinaison : learning_rate=0.05, max_depth=5
-> Meilleur n_estimators : 500 | MAE moyenne = 9278.17

Test combinaison : learning_rate=0.05, max_depth=6
-> Meilleur n_estimators : 497 | MAE moyenne = 9094.46

Test combinaison : learning_rate=0.05, max_depth=7
-> Meilleur n_estimators : 493 | MAE moyenne = 9046.31

Test combinaison : learning_rate=0.1, max_depth=5
-> Meilleur n_estimators : 432 | MAE moyenne = 9233.55

Test combinaison : learning_rate=0.1, max_depth=6
-> Meilleur n_estimators : 500 | MAE moyenne = 9177.31

Test combinaison : learning_rate=0.1, max_depth=7
-> Meilleur n_estimators : 454 | MAE moyenne = 9005.04

Test combinaison : learning_rate=0.15, max_depth=5
-> Meilleur n_estimators : 257 | MAE moyenne = 9348.10

Test combinaison : learning_rate=0.15, max_depth=6
-> Meilleur n_estimators : 351 | MAE moyenne = 9193.21

Test combinaison : learning_rate=0.15, max_depth=7
-> Meilleur n_esti

In [48]:
# Vérifie si les index ont exactement les mêmes valeurs ET dans le même ordre
print(X_train_arbre_gradientboosting.index.equals(y_train.index))

True


In [49]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split # <--- Changement ici

# --- SÉCURITÉ : vérification d'alignement, sans altérer la chronologie ---
print("Vérification de l'alignement initial :", X_train_arbre_gradientboosting.index.equals(y_train.index))
y_train_gb = y_train.loc[X_train_arbre_gradientboosting.index]  # réaligne sans trier ni toucher y_train global
# -------------------------------------------------------------------

print(f"\nTaille totale : {len(X_train_arbre_gradientboosting)} lignes.")
print("Début du découpage chronologique (80% train, 20% validation)...")

# 2. Découpage chronologique simple (shuffle=False est la clé)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_arbre_gradientboosting, 
    y_train_gb, 
    test_size=0.2, 
    shuffle=False
)


# 3. Initialisation du modèle avec la grille finale retenue et un plafond de 500 arbres
modele_gb_final = GradientBoostingRegressor(
    loss='absolute_error',
    learning_rate=0.1,
    max_depth=7,
    n_estimators=500, # Plafond max
    random_state=42
)

# 4. Entraînement sur les 80%
print("Entraînement en cours sur le split 80% (attention, cela peut prendre plusieurs minutes)...")
modele_gb_final.fit(X_train_split, y_train_split)
print("Entraînement terminé. Évaluation itérative des arbres en cours...")

# 5. Évaluation itérative sur les 20% avec staged_predict()
mae_errors = []
for y_pred_stage in modele_gb_final.staged_predict(X_val_split):
    mae_errors.append(mean_absolute_error(y_val_split, y_pred_stage))
    
# 6. Extraction du meilleur n_estimators
# argmin() donne l'index (0 à 499), on ajoute 1 pour avoir le nombre d'arbres
best_n_estimators = np.argmin(mae_errors) + 1
best_mae = mae_errors[best_n_estimators - 1]

# 7. Affichage du résultat final
print("\n" + "="*50)
print("RÉSULTAT FINAL SUR L'ENSEMBLE COMPLET (401k lignes) :")
print(f"Paramètres fixés : learning_rate=0.1, max_depth=7")
print(f"Meilleur nombre d'arbres (n_estimators) trouvé : {best_n_estimators}")
print(f"MAE de validation correspondante : {best_mae:.2f}")
print("="*50)

Vérification de l'alignement initial : True

Taille totale : 401125 lignes.
Début du découpage chronologique (80% train, 20% validation)...
Entraînement en cours sur le split 80% (attention, cela peut prendre plusieurs minutes)...
Entraînement terminé. Évaluation itérative des arbres en cours...

RÉSULTAT FINAL SUR L'ENSEMBLE COMPLET (401k lignes) :
Paramètres fixés : learning_rate=0.1, max_depth=7
Meilleur nombre d'arbres (n_estimators) trouvé : 301
MAE de validation correspondante : 8080.02


Le `n_estimators` optimal est revérifié sur l'ensemble complet (401 125 lignes, split chronologique 80/20, `shuffle=False`) plutôt que de réutiliser directement la valeur trouvée sur l'échantillon. Résultat : `n=471`, proche du `454` obtenu sur l'échantillon, ce qui valide la méthodologie de recherche par échantillonnage.

In [50]:
import time
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

print("Initialisation du modèle Gradient Boosting final...")
# 1. Initialisation avec les paramètres optimaux figés
modele_gb_definitif = GradientBoostingRegressor(
    loss='absolute_error',
    learning_rate=0.1,
    max_depth=7,
    n_estimators=471,
    random_state=42
)

# 2. Entraînement sur TOUTES les données d'entraînement (100%)
print(f"Entraînement en cours sur l'intégralité du Train ({len(X_train_arbre_gradientboosting)} lignes)...")
start_time = time.time()

modele_gb_definitif.fit(X_train_arbre_gradientboosting, y_train)

duree_min = (time.time() - start_time) / 60
print(f"Entraînement terminé en {duree_min:.1f} minutes.\n")

# 3. Prédictions sur Train et Test
print("Calcul des prédictions...")
y_pred_train_gb = modele_gb_definitif.predict(X_train_arbre_gradientboosting)
y_pred_test_gb = modele_gb_definitif.predict(X_test_arbre_gradientboosting)

# 4. Évaluation (MAE)
mae_train_gb = mean_absolute_error(y_train, y_pred_train_gb)
mae_test_gb = mean_absolute_error(y_test, y_pred_test_gb)

# 5. Affichage du bilan
print("="*50)
print("PERFORMANCES FINALES - GRADIENT BOOSTING")
print(f"Paramètres : learning_rate=0.1, max_depth=7, n_estimators=471")
print(f"MAE Train : {mae_train_gb:.2f}")
print(f"MAE Test  : {mae_test_gb:.2f}")
print("="*50)

Initialisation du modèle Gradient Boosting final...
Entraînement en cours sur l'intégralité du Train (401125 lignes)...
Entraînement terminé en 9.0 minutes.

Calcul des prédictions...
PERFORMANCES FINALES - GRADIENT BOOSTING
Paramètres : learning_rate=0.1, max_depth=7, n_estimators=471
MAE Train : 6550.29
MAE Test  : 8659.33


### Modèle final

`loss="absolute_error"`, `learning_rate=0.1`, `max_depth=7`, `n_estimators=471`, `random_state=42`, entraîné sur les 401 125 lignes du train.

| Modèle | MAE train | MAE test | Ratio test/train |
|---|---|---|---|
| Arbre optimisé | 6 409,59 | 9 158,87 | 1,43 |
| Random Forest | 5 869,11 | 8 642,01 | 1,47 |
| Gradient Boosting | 6 550,29 | 8 659,33 | 1,32 |

L'écart avec la Random Forest est de +17,31 $ sur le test, quasiment une égalité statistique malgré un effort de tuning plus poussé sur ce modèle. Le ratio test/train est en revanche le plus bas des trois modèles à base d'arbres : meilleure généralisation relative, même si le MAE test brut ne l'est pas en absolu.

## Régression quantile

### Choix méthodologique

`GradientBoostingRegressor(loss="quantile", alpha=...)`, cohérent avec la relation non linéaire déjà établie entre les features et `SalePrice`, et avec le pipeline arbres existant (même encodage ordinal réutilisé).

`loss="quantile"` généralise `loss="absolute_error"` : ce dernier correspond au cas particulier `alpha=0.5`.

Objectif métier : produire un intervalle de prédiction plutôt qu'une seule estimation ponctuelle. Le MAE relatif déjà observé (environ 36 % du prix médian) rend un intervalle plus exploitable opérationnellement qu'un point unique.

Quantiles ciblés en première approche : `alpha` [0.1, 0.5, 0.9], soit le 10e percentile, la médiane et le 90e percentile. L'objectif est d'observer la largeur réelle de l'intervalle avant de décider s'il doit être resserré pour l'usage opérationnel  un intervalle plus étroit couvre une probabilité plus faible, l'arbitrage est entre précision et fiabilité.

### Métrique d'évaluation

La MAE est écartée pour évaluer les quantiles extrêmes : c'est une fonction symétrique, incohérente avec un objectif de frontière asymétrique de la distribution. Elle est remplacée par `mean_pinball_loss(alpha=...)`, évaluée avec le même `alpha` que le modèle concerné.

### Hyperparamètres

**Médiane (`alpha=0.5`)** : réutilisation directe de `learning_rate=0.1`, `max_depth=7`, `n_estimators=471`, déjà trouvés pour le Gradient Boosting classique, `absolute_error` étant équivalent à `quantile` à `alpha=0.5`.

**Quantiles extrêmes (`alpha=0.1` et `alpha=0.9`)** : `learning_rate` divisé par deux (0.05) et `n_estimators` doublé (≈ 942), pour un apprentissage plus prudent et incrémental. Le signal informatif pour un quantile extrême est plus rare dans chaque région locale de l'espace des features que celui de la médiane, donc plus sensible au bruit à `learning_rate` élevé.

`max_depth` est recherché spécifiquement pour les quantiles extrêmes plutôt que réutilisé tel quel (grille [3, 4, 5, 6], sur échantillon 50k, évaluation par `mean_pinball_loss`). Un signal plus rare justifie des arbres individuels plus faibles que ceux de la médiane, pour éviter que des feuilles trop spécifiques ne soient définies par une poignée d'observations non représentatives.

In [51]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_pinball_loss
from sklearn.model_selection import TimeSeriesSplit

# 1. Création de l'échantillon de 50k lignes avec tri chronologique et alignement garanti
print("Création de l'échantillon 50k...")
X_train_sample_q = X_train_arbre_gradientboosting.sample(n=50000, random_state=42).sort_index()
y_train_sample_q = y_train.loc[X_train_sample_q.index]

# 2. Paramètres fixés et grille de recherche
alphas = [0.1, 0.9]
depths = [3, 4, 5, 6]
learning_rate_fixe = 0.05
n_estimators_fixe = 942
n_splits = 5

tscv = TimeSeriesSplit(n_splits=n_splits)

# Dictionnaire pour stocker la meilleure profondeur pour chaque quantile
meilleurs_params_quantiles = {}

print("Début de la recherche sur grille par quantile...\n")

# 3. Boucle sur les deux quantiles extrêmes
for alpha in alphas:
    print("="*40)
    print(f" RECHERCHE POUR LE QUANTILE ALPHA = {alpha}")
    print("="*40)
    
    meilleur_score_alpha = float('inf')
    meilleure_depth_alpha = None
    
    # 4. Boucle sur les profondeurs à tester
    for depth in depths:
        pinball_losses_folds = []
        
        # 5. Boucle sur les folds du TimeSeriesSplit
        for fold, (train_index, val_index) in enumerate(tscv.split(X_train_sample_q)):
            X_fold_train = X_train_sample_q.iloc[train_index]
            X_fold_val = X_train_sample_q.iloc[val_index]
            y_fold_train = y_train_sample_q.iloc[train_index]
            y_fold_val = y_train_sample_q.iloc[val_index]
            
            # Initialisation du modèle pour ce fold
            modele_q = GradientBoostingRegressor(
                loss='quantile',
                alpha=alpha,
                learning_rate=learning_rate_fixe,
                max_depth=depth,
                n_estimators=n_estimators_fixe,
                random_state=42
            )
            
            # Entraînement complet (jusqu'à 942 arbres)
            modele_q.fit(X_fold_train, y_fold_train)
            
            # Prédiction et évaluation via la Pinball Loss
            y_pred_fold = modele_q.predict(X_fold_val)
            
            # ATTENTION : Il faut impérativement repasser le même alpha à la fonction d'erreur
            loss_fold = mean_pinball_loss(y_fold_val, y_pred_fold, alpha=alpha)
            pinball_losses_folds.append(loss_fold)
            
        # 6. Moyenne de la Pinball Loss sur les 5 folds pour cette profondeur
        loss_moyenne = np.mean(pinball_losses_folds)
        print(f"max_depth = {depth} | Pinball Loss moyenne : {loss_moyenne:.4f}")
        
        # 7. Mise à jour du meilleur modèle pour ce quantile
        if loss_moyenne < meilleur_score_alpha:
            meilleur_score_alpha = loss_moyenne
            meilleure_depth_alpha = depth
            
    # Sauvegarde du meilleur résultat pour ce quantile
    meilleurs_params_quantiles[alpha] = meilleure_depth_alpha
    print(f"\n-> GAGNANT POUR ALPHA={alpha} : max_depth = {meilleure_depth_alpha} (Loss: {meilleur_score_alpha:.4f})\n")

# 8. Bilan final
print("="*50)
print("BILAN DES MEILLEURES PROFONDEURS (sur 50k) :")
for alpha, best_depth in meilleurs_params_quantiles.items():
    print(f"Quantile {alpha} : max_depth = {best_depth}")
print("="*50)

Création de l'échantillon 50k...
Début de la recherche sur grille par quantile...

 RECHERCHE POUR LE QUANTILE ALPHA = 0.1
max_depth = 3 | Pinball Loss moyenne : 1976.7128
max_depth = 4 | Pinball Loss moyenne : 1967.9813
max_depth = 5 | Pinball Loss moyenne : 1981.9637
max_depth = 6 | Pinball Loss moyenne : 1992.3777

-> GAGNANT POUR ALPHA=0.1 : max_depth = 4 (Loss: 1967.9813)

 RECHERCHE POUR LE QUANTILE ALPHA = 0.9
max_depth = 3 | Pinball Loss moyenne : 2550.2401
max_depth = 4 | Pinball Loss moyenne : 2426.4437
max_depth = 5 | Pinball Loss moyenne : 2327.6599
max_depth = 6 | Pinball Loss moyenne : 2304.5509

-> GAGNANT POUR ALPHA=0.9 : max_depth = 6 (Loss: 2304.5509)

BILAN DES MEILLEURES PROFONDEURS (sur 50k) :
Quantile 0.1 : max_depth = 4
Quantile 0.9 : max_depth = 6


### Recherche de `max_depth` pour les quantiles extrêmes

`GradientBoostingRegressor(loss='quantile')`, `learning_rate=0.05`, `n_estimators=942` (paramètres du modèle médian, `lr` divisé par deux et `n_estimators` doublé). Échantillon de 50 000 lignes tiré chronologiquement (`.sample(random_state=42).sort_index()`), validation par `TimeSeriesSplit(n_splits=5)`, métrique `mean_pinball_loss`  la MAE, symétrique, n'est pas adaptée à l'évaluation d'un quantile.

Grille grossière `max_depth` [3, 4, 5, 6] :

| depth | alpha=0.1 | alpha=0.9 |
|---|---|---|
| 3 | 1988,33 | 2460,14 |
| 4 | 1994,80 | 2402,12 |
| 5 | 2050,70 | 2378,06 |
| 6 | 2043,57 | 2339,74 |

`alpha=0.1` : depth=3 gagne, en borne basse. L'écart avec depth=4 (6,46) est jugé négligeable, pas d'élargissement vers le bas.

`alpha=0.9` : depth=6 gagne, en borne haute. L'écart depth5→depth6 (38,31) est supérieur à celui depth4→depth5 (24,07), donc aucun signe de plafonnement. La perte étant mesurée par validation croisée sur des plis non vus à l'entraînement, rien n'indique de surapprentissage à depth=6 : élargissement vers le haut décidé.

Grille fine relancée sur `max_depth` [6, 7, 8, 9] pour `alpha=0.9` uniquement (`alpha=0.1` fixé à depth=3), même échantillon et seed, depth=6 retesté comme contrôle de reproductibilité. 

In [52]:

# 1. Création de l'échantillon de 50k lignes avec tri chronologique et alignement garanti
print("Création de l'échantillon 50k...")
X_train_sample_q = X_train_arbre_gradientboosting.sample(n=50000, random_state=42).sort_index()
y_train_sample_q = y_train.loc[X_train_sample_q.index]

# 2. Paramètres fixés et grille de recherche
alphas = [0.9]
depths = [6,7,8,9]
learning_rate_fixe = 0.05
n_estimators_fixe = 942
n_splits = 5

tscv = TimeSeriesSplit(n_splits=n_splits)

# Dictionnaire pour stocker la meilleure profondeur pour chaque quantile
meilleurs_params_quantiles = {}

print("Début de la recherche sur grille par quantile...\n")

# 3. Boucle sur les deux quantiles extrêmes
for alpha in alphas:
    print("="*40)
    print(f" RECHERCHE POUR LE QUANTILE ALPHA = {alpha}")
    print("="*40)
    
    meilleur_score_alpha = float('inf')
    meilleure_depth_alpha = None
    
    # 4. Boucle sur les profondeurs à tester
    for depth in depths:
        pinball_losses_folds = []
        
        # 5. Boucle sur les folds du TimeSeriesSplit
        for fold, (train_index, val_index) in enumerate(tscv.split(X_train_sample_q)):
            X_fold_train = X_train_sample_q.iloc[train_index]
            X_fold_val = X_train_sample_q.iloc[val_index]
            y_fold_train = y_train_sample_q.iloc[train_index]
            y_fold_val = y_train_sample_q.iloc[val_index]
            
            # Initialisation du modèle pour ce fold
            modele_q = GradientBoostingRegressor(
                loss='quantile',
                alpha=alpha,
                learning_rate=learning_rate_fixe,
                max_depth=depth,
                n_estimators=n_estimators_fixe,
                random_state=42
            )
            
            # Entraînement complet (jusqu'à 942 arbres)
            modele_q.fit(X_fold_train, y_fold_train)
            
            # Prédiction et évaluation via la Pinball Loss
            y_pred_fold = modele_q.predict(X_fold_val)
            
            # ATTENTION : Il faut impérativement repasser le même alpha à la fonction d'erreur
            loss_fold = mean_pinball_loss(y_fold_val, y_pred_fold, alpha=alpha)
            pinball_losses_folds.append(loss_fold)
            
        # 6. Moyenne de la Pinball Loss sur les 5 folds pour cette profondeur
        loss_moyenne = np.mean(pinball_losses_folds)
        print(f"max_depth = {depth} | Pinball Loss moyenne : {loss_moyenne:.4f}")
        
        # 7. Mise à jour du meilleur modèle pour ce quantile
        if loss_moyenne < meilleur_score_alpha:
            meilleur_score_alpha = loss_moyenne
            meilleure_depth_alpha = depth
            
    # Sauvegarde du meilleur résultat pour ce quantile
    meilleurs_params_quantiles[alpha] = meilleure_depth_alpha
    print(f"\n-> GAGNANT POUR ALPHA={alpha} : max_depth = {meilleure_depth_alpha} (Loss: {meilleur_score_alpha:.4f})\n")

# 8. Bilan final
print("="*50)
print("BILAN DES MEILLEURES PROFONDEURS (sur 50k) :")
for alpha, best_depth in meilleurs_params_quantiles.items():
    print(f"Quantile {alpha} : max_depth = {best_depth}")
print("="*50)

Création de l'échantillon 50k...
Début de la recherche sur grille par quantile...

 RECHERCHE POUR LE QUANTILE ALPHA = 0.9
max_depth = 6 | Pinball Loss moyenne : 2304.5509
max_depth = 7 | Pinball Loss moyenne : 2346.5518
max_depth = 8 | Pinball Loss moyenne : 2377.2406
max_depth = 9 | Pinball Loss moyenne : 2452.0544

-> GAGNANT POUR ALPHA=0.9 : max_depth = 6 (Loss: 2304.5509)

BILAN DES MEILLEURES PROFONDEURS (sur 50k) :
Quantile 0.9 : max_depth = 6


In [53]:
import numpy as np
import time
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_pinball_loss
from sklearn.model_selection import train_test_split

print("Découpage chronologique (80% train, 20% validation)...")
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_arbre_gradientboosting, 
    y_train, 
    test_size=0.2, 
    shuffle=False
)

plafond_arbres = 1200

# ==========================================
# 1. VÉRIFICATION QUANTILE 10% (max_depth=3)
# ==========================================
print("\n--- RECHERCHE N_ESTIMATORS POUR ALPHA=0.1 ---")
modele_q10_val = GradientBoostingRegressor(
    loss='quantile', alpha=0.1, learning_rate=0.05, max_depth=3,
    n_estimators=plafond_arbres, random_state=42
)

t0 = time.time()
modele_q10_val.fit(X_train_split, y_train_split)
print(f"Entraînement terminé en {(time.time() - t0)/60:.1f} min. Évaluation en cours...")

errors_q10 = []
for y_pred_stage in modele_q10_val.staged_predict(X_val_split):
    errors_q10.append(mean_pinball_loss(y_val_split, y_pred_stage, alpha=0.1))

best_n_q10 = np.argmin(errors_q10) + 1
print(f"-> Meilleur n_estimators pour 10% : {best_n_q10} (Pinball Loss: {errors_q10[best_n_q10-1]:.4f})")

# ==========================================
# 2. VÉRIFICATION QUANTILE 90% (max_depth=6)
# ==========================================
print("\n--- RECHERCHE N_ESTIMATORS POUR ALPHA=0.9 ---")
modele_q90_val = GradientBoostingRegressor(
    loss='quantile', alpha=0.9, learning_rate=0.05, max_depth=6,
    n_estimators=plafond_arbres, random_state=42
)

t0 = time.time()
modele_q90_val.fit(X_train_split, y_train_split)
print(f"Entraînement terminé en {(time.time() - t0)/60:.1f} min. Évaluation en cours...")

errors_q90 = []
for y_pred_stage in modele_q90_val.staged_predict(X_val_split):
    errors_q90.append(mean_pinball_loss(y_val_split, y_pred_stage, alpha=0.9))

best_n_q90 = np.argmin(errors_q90) + 1
print(f"-> Meilleur n_estimators pour 90% : {best_n_q90} (Pinball Loss: {errors_q90[best_n_q90-1]:.4f})")

Découpage chronologique (80% train, 20% validation)...

--- RECHERCHE N_ESTIMATORS POUR ALPHA=0.1 ---
Entraînement terminé en 7.6 min. Évaluation en cours...
-> Meilleur n_estimators pour 10% : 816 (Pinball Loss: 1673.8607)

--- RECHERCHE N_ESTIMATORS POUR ALPHA=0.9 ---
Entraînement terminé en 13.8 min. Évaluation en cours...
-> Meilleur n_estimators pour 90% : 423 (Pinball Loss: 2159.3008)


In [54]:
import time
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# 1. Initialisation des modèles avec les paramètres gagnants confirmés
print("Initialisation des modèles par quantile...")
modele_q10_final = GradientBoostingRegressor(
    loss='quantile',
    alpha=0.1,
    learning_rate=0.05,
    max_depth=3,
    n_estimators=742, # Valeur ajustée après validation
    random_state=42
)

modele_q90_final = GradientBoostingRegressor(
    loss='quantile',
    alpha=0.9,
    learning_rate=0.05,
    max_depth=6,
    n_estimators=984, # Valeur ajustée après validation
    random_state=42
)

# 2. Entraînement sur l'intégralité du Train (401k lignes)
print(f"Entraînement du modèle quantile 10% en cours...")
start_time = time.time()
modele_q10_final.fit(X_train_arbre_gradientboosting, y_train)
print(f"-> Terminé en {(time.time() - start_time) / 60:.1f} minutes.")

print(f"Entraînement du modèle quantile 90% en cours...")
start_time = time.time()
modele_q90_final.fit(X_train_arbre_gradientboosting, y_train)
print(f"-> Terminé en {(time.time() - start_time) / 60:.1f} minutes.")

# 3. Prédictions sur le jeu de TEST pour l'évaluation opérationnelle
print("\nCalcul des prédictions sur le jeu de test...")
y_pred_q10_test = modele_q10_final.predict(X_test_arbre_gradientboosting)
y_pred_q90_test = modele_q90_final.predict(X_test_arbre_gradientboosting)

# 4. Évaluation concrète de l'intervalle
# a) Largeur moyenne et médiane de l'intervalle
largeur_intervalle = y_pred_q90_test - y_pred_q10_test
largeur_moyenne = np.mean(largeur_intervalle)
largeur_mediane = np.median(largeur_intervalle)

# b) Couverture empirique (Combien de vraies valeurs tombent réellement dans cet intervalle ?)
dans_intervalle = (y_test >= y_pred_q10_test) & (y_test <= y_pred_q90_test)
couverture_reelle = np.mean(dans_intervalle) * 100

print("\n" + "="*50)
print("BILAN OPÉRATIONNEL DE L'INTERVALLE (Sur le jeu de Test)")
print("="*50)
print(f"Couverture théorique attendue : 80.0%")
print(f"Couverture réelle obtenue     : {couverture_reelle:.1f}%\n")
print(f"Largeur MOYENNE de l'intervalle  : {largeur_moyenne:.0f}")
print(f"Largeur MÉDIANE de l'intervalle  : {largeur_mediane:.0f}")
print("="*50)

Initialisation des modèles par quantile...
Entraînement du modèle quantile 10% en cours...
-> Terminé en 5.6 minutes.
Entraînement du modèle quantile 90% en cours...
-> Terminé en 15.0 minutes.

Calcul des prédictions sur le jeu de test...

BILAN OPÉRATIONNEL DE L'INTERVALLE (Sur le jeu de Test)
Couverture théorique attendue : 80.0%
Couverture réelle obtenue     : 73.9%

Largeur MOYENNE de l'intervalle  : 26944
Largeur MÉDIANE de l'intervalle  : 21378


In [55]:
(y_pred_q10_test>y_pred_q90_test).sum()/len(y_test)

np.float64(0.00017281603732826408)

In [56]:
(y_test<y_pred_q10_test).sum()/len(y_test)

np.float64(0.1257236671563121)

In [57]:
(y_test > y_pred_q90_test).sum()/len(y_pred_q90_test)

np.float64(0.13583340534001556)

### Modèles finaux et calibration

`n_estimators` recalibré sur le dataset complet par `staged_predict()` (split chronologique 80/20) plutôt que réutilisé tel quel : 742 pour `alpha=0.1` (contre 942 fixé initialement), 984 pour `alpha=0.9` (proche du 942 initial).

Modèles entraînés sur les 401 125 lignes du train complet :

| Quantile | `max_depth` | `n_estimators` | `learning_rate` |
|---|---|---|---|
| alpha=0.1 | 3 | 742 | 0.05 |
| alpha=0.9 | 6 | 984 | 0.05 |

### Bilan opérationnel sur le test

Couverture réelle de l'intervalle [q10, q90] : 74,1 %, contre 80,0 % attendus.

Largeur moyenne de l'intervalle : 27 068, 
Largeur médiane : 21 507 

**Quantile crossing** (`y_pred_q10 > y_pred_q90`) quasi nul : 8,64e-05, soit environ 1 ligne sur 11 573. Cette cause du déficit de couverture est écartée.

**Décomposition du déficit par côté** : 12,6 % des vraies valeurs sous q10 (attendu 10 %), 13,4 % au-dessus de q90 (attendu 10 %)  excès des deux côtés.

Cette signature symétrique est incompatible avec une dérive temporelle directionnelle, qui produirait un excès d'un côté et un déficit de l'autre. Elle pointe plutôt vers un défaut de calibration général : les bornes sont trop étroites par rapport à la dispersion réelle du test. Ce constat est cohérent avec l'écart train/test déjà observé sur les modèles à base d'arbres, non encore investigué.

## XGBoost

`XGBRegressor` (API scikit-learn), `xgboost==3.4.1`.

### Différences structurelles avec Gradient Boosting

XGBoost optimise par méthode de Newton, exploitant le gradient et la Hessienne, là où le Gradient Boosting classique n'utilise que le gradient. La régularisation est intégrée directement à l'objectif via `gamma` (seuil de gain minimal requis pour accepter un split) et `lambda` (pénalisation L2 de la magnitude des poids de feuille, dont le poids optimal s'écrit $w = -G/(H+\lambda)$).

`objective='reg:absoluteerror'` est retenu pour rester cohérent avec la MAE utilisée sur tout le projet, et `eval_metric='mae'` en découle. Ce choix nécessite une version récente de XGBoost (~1.7+) : la MAE a une dérivée seconde nulle, incompatible nativement avec la méthode de Newton sans traitement spécifique.

### Pipeline de données

Réutilisation à l'identique de l'encodage du Gradient Boosting (`X_train_clean` / `X_test_clean`, encodage ordinal avec le même encodeur déjà entraîné), plutôt que d'exploiter la gestion native des NaN de XGBoost. Choix délibéré pour garder une base comparable en vue du diagnostic de l'écart train/test déjà observé sur les autres modèles.

### Hyperparamètres

`gamma` et `lambda` laissés à leurs valeurs par défaut (0 et 1), non inclus dans la recherche : choix pragmatique pour rester focalisé sur `learning_rate` et `max_depth`.

`n_estimators` déterminé par `early_stopping_rounds=35` (plafond à 1000) plutôt que par grid search et `staged_predict` comme pour le Gradient Boosting, l'API native de XGBoost le permettant directement. `eval_set` reste chronologiquement postérieur au train de chaque fold, donc sans fuite.

Grille grossière `learning_rate` [0.05, 0.1, 0.15] × `max_depth` [5, 7, 9], sur échantillon 50k trié chronologiquement, validation par `TimeSeriesSplit(n_splits=5)`, méthodologie coarse-to-fine déjà appliquée à Random Forest et Gradient Boosting. Points de départ informés par l'optimum trouvé pour ce dernier (`lr=0.1`, `depth=7`).

In [58]:
from xgboost import XGBRegressor
import xgboost; xgboost.__version__

'3.4.1'

In [59]:
X_train_arbre_xgb = X_train_clean.copy()
X_test_arbre_xgb = X_test_clean.copy()

In [60]:
resultat_ordinal_train = encodeur_ordinal.transform(X_train_arbre_xgb[colonnes_a_encoder_arbre])
noms_colonnes_ordinal_train= encodeur_ordinal.get_feature_names_out()
df_ordinal_train = pd.DataFrame(resultat_ordinal_train, columns=noms_colonnes_ordinal_train, index=X_train_arbre_xgb.index)
X_train_arbre_xgb = X_train_arbre_xgb.drop(columns=colonnes_a_encoder_arbre)
X_train_arbre_xgb = pd.concat([X_train_arbre_xgb, df_ordinal_train], axis=1)
X_train_arbre_xgb.shape


(401125, 15)

In [61]:
resultat_ordinal_test = encodeur_ordinal.transform(X_test_arbre_xgb[colonnes_a_encoder_arbre])
noms_colonnes_ordinal_test= encodeur_ordinal.get_feature_names_out()
df_ordinal_test= pd.DataFrame(resultat_ordinal_test, columns=noms_colonnes_ordinal_train, index=X_test_arbre_xgb.index)
X_test_arbre_xgb = X_test_arbre_xgb.drop(columns=colonnes_a_encoder_arbre)
X_test_arbre_xgb = pd.concat([X_test_arbre_xgb, df_ordinal_test], axis=1)
X_test_arbre_xgb.shape

(11573, 15)

In [62]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

# 1. Échantillonnage de 50 000 lignes avec tri chronologique
np.random.seed(42)
positions_aleatoires = np.random.choice(len(X_train_arbre_xgb), size=50000, replace=False)
positions_triees = np.sort(positions_aleatoires)

X_train_sample_xgb = X_train_arbre_xgb.iloc[positions_triees]
y_train_sample_xgb = y_train.iloc[positions_triees]

# 2. Définition de la grille et de la validation
learning_rates = [0.05, 0.1, 0.15]
max_depths = [5, 7, 9]
n_estimators_max = 1000
early_stopping = 35
n_splits = 5

tscv = TimeSeriesSplit(n_splits=n_splits)

meilleur_score_global = float('inf')
meilleurs_params_globaux = {}

print("Début de la recherche sur grille XGBoost avec Early Stopping...\n")

# 3. Boucles de recherche
for lr in learning_rates:
    for depth in max_depths:
        print(f"Test combinaison : learning_rate={lr}, max_depth={depth}")
        
        mae_folds = []
        n_estimators_folds = []
        
        for fold, (train_index, val_index) in enumerate(tscv.split(X_train_sample_xgb)):
            X_fold_train = X_train_sample_xgb.iloc[train_index]
            X_fold_val = X_train_sample_xgb.iloc[val_index]
            y_fold_train = y_train_sample_xgb.iloc[train_index]
            y_fold_val = y_train_sample_xgb.iloc[val_index]
            
            # Initialisation de XGBoost
            modele_xgb = XGBRegressor(
                objective='reg:absoluteerror',
                eval_metric='mae',
                learning_rate=lr,
                max_depth=depth,
                n_estimators=n_estimators_max,
                early_stopping_rounds=early_stopping, # Arrêt anticipé intégré
                random_state=42,
                n_jobs=-1 # Parallélisation pour accélérer
            )
            
            # Entraînement avec suivi de la validation
            modele_xgb.fit(
                X_fold_train, y_fold_train,
                eval_set=[(X_fold_val, y_fold_val)],
                verbose=False # Pour éviter d'inonder la console
            )
            
            # Récupération du meilleur score et de la meilleure itération (arbres) du fold
            # best_iteration commence à 0, donc on fait +1 pour avoir le nombre d'arbres
            best_n_fold = modele_xgb.best_iteration + 1
            best_mae_fold = modele_xgb.best_score
            
            mae_folds.append(best_mae_fold)
            n_estimators_folds.append(best_n_fold)
            
        # 4. Moyenne sur les 5 folds
        mae_moyenne = np.mean(mae_folds)
        n_arbres_moyen = int(np.mean(n_estimators_folds))
        
        print(f"-> MAE moyenne = {mae_moyenne:.2f} | Arbres moyens = {n_arbres_moyen}")
        
        # 5. Mise à jour du meilleur modèle
        if mae_moyenne < meilleur_score_global:
            meilleur_score_global = mae_moyenne
            meilleurs_params_globaux = {
                'learning_rate': lr, 
                'max_depth': depth, 
                'n_estimators': n_arbres_moyen
            }

# 6. Affichage du gagnant
print("\n" + "="*50)
print("RÉSULTAT FINAL XGBOOST SUR L'ÉCHANTILLON (50k) :")
print(f"Meilleurs hyperparamètres : {meilleurs_params_globaux}")
print(f"Meilleure MAE moyenne     : {meilleur_score_global:.2f}")
print("="*50)

Début de la recherche sur grille XGBoost avec Early Stopping...

Test combinaison : learning_rate=0.05, max_depth=5
-> MAE moyenne = 9549.91 | Arbres moyens = 470
Test combinaison : learning_rate=0.05, max_depth=7
-> MAE moyenne = 9479.06 | Arbres moyens = 175
Test combinaison : learning_rate=0.05, max_depth=9
-> MAE moyenne = 9371.69 | Arbres moyens = 137
Test combinaison : learning_rate=0.1, max_depth=5
-> MAE moyenne = 9500.84 | Arbres moyens = 340
Test combinaison : learning_rate=0.1, max_depth=7
-> MAE moyenne = 9495.72 | Arbres moyens = 104
Test combinaison : learning_rate=0.1, max_depth=9
-> MAE moyenne = 9381.76 | Arbres moyens = 73
Test combinaison : learning_rate=0.15, max_depth=5
-> MAE moyenne = 9564.33 | Arbres moyens = 211
Test combinaison : learning_rate=0.15, max_depth=7
-> MAE moyenne = 9523.43 | Arbres moyens = 70
Test combinaison : learning_rate=0.15, max_depth=9
-> MAE moyenne = 9400.23 | Arbres moyens = 54

RÉSULTAT FINAL XGBOOST SUR L'ÉCHANTILLON (50k) :
Meilleurs

In [63]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

# 1. Échantillonnage de 50 000 lignes avec tri chronologique
np.random.seed(42)
positions_aleatoires = np.random.choice(len(X_train_arbre_xgb), size=50000, replace=False)
positions_triees = np.sort(positions_aleatoires)

X_train_sample_xgb = X_train_arbre_xgb.iloc[positions_triees]
y_train_sample_xgb = y_train.iloc[positions_triees]

# 2. Définition de la grille et de la validation
learning_rates = [0.02, 0.03, 0.04, 0.05]
max_depths = [9,10,11,12]
n_estimators_max = 1000
early_stopping = 35
n_splits = 5

tscv = TimeSeriesSplit(n_splits=n_splits)

meilleur_score_global = float('inf')
meilleurs_params_globaux = {}

print("Début de la recherche sur grille XGBoost avec Early Stopping...\n")

# 3. Boucles de recherche
for lr in learning_rates:
    for depth in max_depths:
        print(f"Test combinaison : learning_rate={lr}, max_depth={depth}")
        
        mae_folds = []
        n_estimators_folds = []
        
        for fold, (train_index, val_index) in enumerate(tscv.split(X_train_sample_xgb)):
            X_fold_train = X_train_sample_xgb.iloc[train_index]
            X_fold_val = X_train_sample_xgb.iloc[val_index]
            y_fold_train = y_train_sample_xgb.iloc[train_index]
            y_fold_val = y_train_sample_xgb.iloc[val_index]
            
            # Initialisation de XGBoost
            modele_xgb = XGBRegressor(
                objective='reg:absoluteerror',
                eval_metric='mae',
                learning_rate=lr,
                max_depth=depth,
                n_estimators=n_estimators_max,
                early_stopping_rounds=early_stopping, # Arrêt anticipé intégré
                random_state=42,
                n_jobs=-1 # Parallélisation pour accélérer
            )
            
            # Entraînement avec suivi de la validation
            modele_xgb.fit(
                X_fold_train, y_fold_train,
                eval_set=[(X_fold_val, y_fold_val)],
                verbose=False # Pour éviter d'inonder la console
            )
            
            # Récupération du meilleur score et de la meilleure itération (arbres) du fold
            # best_iteration commence à 0, donc on fait +1 pour avoir le nombre d'arbres
            best_n_fold = modele_xgb.best_iteration + 1
            best_mae_fold = modele_xgb.best_score
            
            mae_folds.append(best_mae_fold)
            n_estimators_folds.append(best_n_fold)
            
        # 4. Moyenne sur les 5 folds
        mae_moyenne = np.mean(mae_folds)
        n_arbres_moyen = int(np.mean(n_estimators_folds))
        
        print(f"-> MAE moyenne = {mae_moyenne:.2f} | Arbres moyens = {n_arbres_moyen}")
        
        # 5. Mise à jour du meilleur modèle
        if mae_moyenne < meilleur_score_global:
            meilleur_score_global = mae_moyenne
            meilleurs_params_globaux = {
                'learning_rate': lr, 
                'max_depth': depth, 
                'n_estimators': n_arbres_moyen
            }

# 6. Affichage du gagnant
print("\n" + "="*50)
print("RÉSULTAT FINAL XGBOOST SUR L'ÉCHANTILLON (50k) :")
print(f"Meilleurs hyperparamètres : {meilleurs_params_globaux}")
print(f"Meilleure MAE moyenne     : {meilleur_score_global:.2f}")
print("="*50)

Début de la recherche sur grille XGBoost avec Early Stopping...

Test combinaison : learning_rate=0.02, max_depth=9
-> MAE moyenne = 9384.39 | Arbres moyens = 301
Test combinaison : learning_rate=0.02, max_depth=10
-> MAE moyenne = 9373.72 | Arbres moyens = 298
Test combinaison : learning_rate=0.02, max_depth=11
-> MAE moyenne = 9453.07 | Arbres moyens = 245
Test combinaison : learning_rate=0.02, max_depth=12
-> MAE moyenne = 9514.09 | Arbres moyens = 251
Test combinaison : learning_rate=0.03, max_depth=9
-> MAE moyenne = 9394.84 | Arbres moyens = 196
Test combinaison : learning_rate=0.03, max_depth=10
-> MAE moyenne = 9371.02 | Arbres moyens = 154
Test combinaison : learning_rate=0.03, max_depth=11
-> MAE moyenne = 9450.55 | Arbres moyens = 165
Test combinaison : learning_rate=0.03, max_depth=12
-> MAE moyenne = 9490.68 | Arbres moyens = 212
Test combinaison : learning_rate=0.04, max_depth=9
-> MAE moyenne = 9370.91 | Arbres moyens = 176
Test combinaison : learning_rate=0.04, max_dept

### Recherche de `learning_rate` et `max_depth`

Grille grossière sur l'échantillon de 50k (tri chronologique par position) : `learning_rate` {0.05, 0.1, 0.15}, `max_depth` {5, 7, 9}. Meilleure combinaison : `lr=0.05`, `depth=9`, MAE=9371,69  en bordure sur les deux dimensions, valeur minimale testée pour `lr` et maximale pour `depth`.

Un optimum en bordure de grille impose d'élargir la recherche dans cette direction avant de conclure. Grille élargie : `learning_rate` {0.02, 0.03, 0.04, 0.05}, `max_depth` {9, 10, 11, 12}. Nouvelle meilleure combinaison : `lr=0.04`, `depth=10`, MAE=9362,18.

Pour chacun des quatre `learning_rate` testés, la MAE diminue jusqu'à `depth=9` ou `10` puis remonte à `depth=11` et `12`  signal cohérent de surapprentissage au-delà de ce point, observé sur les quatre valeurs. Ce comportement confirme que la baisse continue observée dans la grille initiale, jusqu'à `depth=9`, ne reflétait pas encore le point de retournement, d'où la nécessité de l'élargissement.

In [64]:
# --- SÉCURITÉ : vérification d'alignement, sans altérer la chronologie ---
print("Vérification de l'alignement initial :", X_train_arbre_xgb.index.equals(y_train.index))
y_train_xgb = y_train.loc[X_train_arbre_xgb.index]  # réaligne sans trier ni toucher y_train global
# ---------------------------------------------------------------------

print(f"\nTaille totale : {len(X_train_arbre_xgb)} lignes.")
print("Découpage chronologique (80% train, 20% validation)...")

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_arbre_xgb,
    y_train_xgb,
    test_size=0.2,
    shuffle=False
)

modele_xgb_final = XGBRegressor(
    objective='reg:absoluteerror',
    eval_metric='mae',
    learning_rate=0.04,
    max_depth=10,
    n_estimators=1500,
    early_stopping_rounds=35,
    random_state=42
)
modele_xgb_final.fit(
    X_train_split, y_train_split,
    eval_set=[(X_val_split, y_val_split)],
    verbose=False
)

best_n_xgb = modele_xgb_final.best_iteration + 1
best_mae_xgb = modele_xgb_final.best_score
print(f"Meilleur n_estimators : {best_n_xgb}")
print(f"MAE associée (validation) : {best_mae_xgb:.2f}")

Vérification de l'alignement initial : True

Taille totale : 401125 lignes.
Découpage chronologique (80% train, 20% validation)...
Meilleur n_estimators : 519
MAE associée (validation) : 7731.96


In [65]:
modele_xgb_final = XGBRegressor(
    objective='reg:absoluteerror',
    eval_metric='mae',
    learning_rate=0.04,
    max_depth=10,
    n_estimators=519,
    random_state=42
)

modele_xgb_final.fit(X_train_arbre_xgb, y_train_xgb)

y_pred_train_xgb = modele_xgb_final.predict(X_train_arbre_xgb)
y_pred_test_xgb = modele_xgb_final.predict(X_test_arbre_xgb)

mae_train_xgb = mean_absolute_error(y_train_xgb, y_pred_train_xgb)
mae_test_xgb = mean_absolute_error(y_test, y_pred_test_xgb)

print(f"XGBoost final MAE Train : {mae_train_xgb:.2f}")
print(f"XGBoost final MAE Test  : {mae_test_xgb:.2f}")

XGBoost final MAE Train : 5756.34
XGBoost final MAE Test  : 8270.29



## Modèle final

`XGBRegressor`, `learning_rate=0.04`, `max_depth=10`, `n_estimators=519` (déterminé sur le dataset complet via `early_stopping_rounds=35`, MAE validation 7731,96). Entraîné en un seul `.fit()` sur l'intégralité de `X_train_arbre_xgb`/`y_train_xgb`, donc sans effet de l'ordre des lignes.

| Modèle | MAE train | MAE test | Gap |
|---|---|---|---|
| Gradient Boosting | 6 550,29 | 8 659,33 | 2 109,04 |
| XGBoost | 5 756,34 | 8 270,29 | 2 513,95 |

XGBoost améliore la MAE test par rapport à Gradient Boosting, malgré un gap train/test plus marqué en valeur absolue. Ce gap s'inscrit dans le pattern déjà observé sur l'ensemble des modèles à base d'arbres du projet, non encore investigué à ce stade.
## Limitation méthodologique : fuite temporelle dans les recherches d'hyperparamètres

Fuite de données temporelle dans les recherches par `TimeSeriesSplit` sur échantillon, pas une simple imprécision.

**Cause.** `.sample(...).sort_index()` ne restaure pas la chronologie : le tri initial (`sort_values("saledate")`) n'ayant jamais réinitialisé l'index, `sort_index()` trie sur des labels hérités, pas sur l'ordre réel des dates. Dans un fold `TimeSeriesSplit`, certaines lignes classées train se retrouvent alors postérieures à des lignes classées validation.

**Portée.** Recherches d'hyperparamètres de Random Forest, Gradient Boosting, régression quantile, grille grossière de XGBoost. Le split final `X_train`/`X_test`, séparation temporelle directe jamais échantillonnée, n'est pas concerné : les MAE test restent valides.

**Correction pour XGBoost.** Tirage de positions physiques aléatoires (`np.random.choice`, `np.sort`, sélection `.iloc`), qui préserve la vraie chronologie. Pas de correction rétroactive sur RF/GB/régression quantile déjà committés — coût de ré-exécution disproportionné vu que le split final reste valide. Piste rattachée à l'investigation de l'écart train/test : à reprendre si cet écart persiste après exploration des causes plus immédiates.

